# Weapon motion measurements · CPU batch
Measurement-only, no similarity percentage or artistic approval. Use a CPU runtime.
No GPU, tracker model, rendering, Drive mount, repository fetch or credentials.
Supply one JSON per action with original source pixels, exact decoded PTS,
evaluated candidate pixels and explicit visibility/uncertainty. Physical anchor
and phase definitions must be independently checked before using any result.
This does not modify the general animation review notebook.


In [ ]:
import types,sys,hashlib,json,tempfile,zipfile
from pathlib import Path
MODULE_SOURCE = '"""Auditable screen-space weapon diagnostics. No tracking, fitting, retiming or score."""\nfrom __future__ import annotations\nimport argparse\nfrom collections import Counter\nfrom fractions import Fraction\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport statistics\n\nSCHEMA = \'rust-duty-weapon-comparison/v1\'\nSTATES = {\'visible\', \'occluded\', \'tracker_failed\', \'uncertain\', \'out_of_frame\', \'missing\'}\nMAX_BYTES = 32 * 1024 * 1024\nMAX_FRAMES = 20000\n\ndef sha256(data):\n    return hashlib.sha256(data).hexdigest()\n\ndef strict_json(data):\n    if len(data) > MAX_BYTES:\n        raise ValueError(\'Input exceeds 32 MiB\')\n    def pairs(items):\n        result = {}\n        for key, value in items:\n            if key in result:\n                raise ValueError(\'Duplicate JSON key: \' + key)\n            result[key] = value\n        return result\n    def reject(value):\n        raise ValueError(\'Nonfinite JSON: \' + value)\n    return json.loads(data, object_pairs_hook=pairs, parse_constant=reject)\n\ndef number(value, name, low=-1e9, high=1e9):\n    if type(value) not in (int, float) or not math.isfinite(value) or not low <= value <= high:\n        raise ValueError(\'Invalid \' + name)\n    return float(value)\n\ndef integer(value, name, low=0, high=2**53):\n    if type(value) is not int or not low <= value <= high:\n        raise ValueError(\'Invalid \' + name)\n    return value\n\ndef rational(value, name):\n    if not isinstance(value, list) or len(value) != 2:\n        raise ValueError(name + \' must be [integer numerator, positive denominator]\')\n    return Fraction(integer(value[0], name, -2**53), integer(value[1], name, 1))\n\ndef ratio(value):\n    return [value.numerator, value.denominator]\n\ndef text(value, name):\n    if not isinstance(value, str) or not value.strip() or len(value) > 2048:\n        raise ValueError(\'Require \' + name)\n    return value\n\ndef artifact(value):\n    text(value.get(\'name\'), \'artifact name\')\n    digest = value.get(\'sha256\')\n    if not isinstance(digest, str) or len(digest) != 64 or any(c not in \'0123456789abcdef\' for c in digest):\n        raise ValueError(\'Require lowercase artifact SHA256\')\n    return value\n\ndef point(value, viewport):\n    if not isinstance(value, dict) or value.get(\'state\') not in STATES:\n        raise ValueError(\'Point needs explicit observability state\')\n    text(value.get(\'method\'), \'point observation/evaluation method\')\n    if \'confidence\' in value:\n        number(value[\'confidence\'], \'model confidence\', 0, 1)\n    if value[\'state\'] == \'visible\':\n        xy = value.get(\'xy\')\n        if not isinstance(xy, list) or len(xy) != 2:\n            raise ValueError(\'Visible point needs xy\')\n        for axis, bound in zip(xy, viewport):\n            number(axis, \'visible pixel coordinate\', 0, bound)\n        number(value.get(\'uncertainty_px\'), \'visible pixel uncertainty\', 0, max(viewport))\n    elif value.get(\'xy\') is not None:\n        if len(value[\'xy\']) != 2:\n            raise ValueError(\'Point xy must have two coordinates\')\n        for axis in value[\'xy\']:\n            number(axis, \'nonvisible pixel coordinate\', -100000, 100000)\n    return value\n\ndef validate(doc):\n    if doc.get(\'schema\') != SCHEMA:\n        raise ValueError(\'Unsupported comparison schema\')\n    text(doc.get(\'action\'), \'independently reviewed action\')\n    text(doc.get(\'rubric_version\'), \'rubric version (pending allowed)\')\n    if doc.get(\'kind\') not in (\'event\', \'loop\'):\n        raise ValueError(\'kind must be event or loop\')\n    source, candidate = doc[\'source\'], doc[\'candidate\']\n    artifact(source[\'artifact\']); artifact(candidate[\'artifact\'])\n    text(candidate.get(\'author\'), \'candidate author\')\n    if candidate.get(\'space\') != \'evaluated-camera-pixels\':\n        raise ValueError(\'Candidate must use evaluated camera projection, not local bones\')\n    for view in (source[\'viewport\'], candidate[\'viewport\']):\n        if not isinstance(view, list) or len(view) != 2:\n            raise ValueError(\'Viewport requires width,height\')\n        for v in view:\n            integer(v, \'viewport dimension\', 1, 16384)\n    tb = rational(source[\'time_base\'], \'source time_base\')\n    if tb <= 0:\n        raise ValueError(\'Time base must be positive\')\n    window = source[\'window_frames\']\n    if len(window) != 2 or any(type(v) is not int or v < 0 for v in window) or not 2 <= window[1] - window[0] <= MAX_FRAMES:\n        raise ValueError(\'Require complete bounded half-open source window\')\n    reg = doc[\'registration\']\n    allowed = {\'mode\', \'source_scale\', \'candidate_scale\', \'source_anchor_pts\', \'candidate_anchor_time\', \'phase_offset\', \'frozen_record\'}\n    if set(reg) != allowed or reg[\'mode\'] != \'frozen-unwarped\':\n        raise ValueError(\'Only one frozen scale and phase mapping; unknown/per-frame registration prohibited\')\n    text(reg[\'frozen_record\'], \'pre-candidate registration record\')\n    ss = number(reg[\'source_scale\'], \'source scale\', .01, 100)\n    cs = number(reg[\'candidate_scale\'], \'candidate scale\', .01, 100)\n    anchor = integer(reg[\'source_anchor_pts\'], \'source anchor PTS\', -2**53)\n    candidate_anchor = rational(reg[\'candidate_anchor_time\'], \'candidate anchor time\')\n    phase = rational(reg[\'phase_offset\'], \'single phase offset\')\n    if abs(phase) > 600:\n        raise ValueError(\'Phase offset out of bounds\')\n    norm = doc[\'normalization\']\n    if norm[\'L_px\'] is not None:\n        number(norm[\'L_px\'], \'fixed reference visible-axis L\', .01, 100000)\n    text(norm[\'definition\'], \'L definition and frozen source evidence\')\n    integer(doc[\'metrics_stride\'], \'fixed metrics stride\', 1, 60)\n    baseline = integer(doc[\'baseline_frame\'], \'fixed pre-event baseline\')\n    if not window[0] <= baseline < window[1]:\n        raise ValueError(\'Baseline must be inside window\')\n    landmarks = doc[\'landmarks\']\n    if not isinstance(landmarks, list) or not 3 <= len(landmarks) <= 16:\n        raise ValueError(\'Require 3..16 identified rigid weapon landmarks\')\n    names = [text(p[\'id\'], \'landmark id\') for p in landmarks]\n    if len(set(names)) != len(names):\n        raise ValueError(\'Duplicate landmark id\')\n    for p in landmarks:\n        text(p.get(\'physical_point\'), \'physical point correspondence\')\n    axis = doc[\'axis_landmarks\']\n    if len(axis) != 2 or len(set(axis)) != 2 or any(p not in names for p in axis):\n        raise ValueError(\'Require two distinct physical axis landmarks\')\n    rows = doc[\'samples\']\n    if not isinstance(rows, list) or len(rows) != window[1] - window[0]:\n        raise ValueError(\'Every source frame required; missing rows cannot hide errors\')\n    previous_pts = None\n    for expected, row in zip(range(*window), rows):\n        if row[\'frame\'] != expected or type(row[\'frame\']) is not int:\n            raise ValueError(\'Source frame indices must cover window consecutively\')\n        pts = integer(row[\'source_pts\'], \'observed source PTS\', -2**53)\n        if previous_pts is not None and pts <= previous_pts:\n            raise ValueError(\'Decoded PTS must strictly increase\')\n        previous_pts = pts\n        expected_time = (pts - anchor) * tb + candidate_anchor + phase\n        if rational(row[\'candidate_time\'], \'candidate evaluation time\') != expected_time:\n            raise ValueError(\'PTS mapping mismatch; speed must stay exactly 1 with one frozen offset\')\n        for side, view in ((\'reference\', source[\'viewport\']), (\'candidate\', candidate[\'viewport\'])):\n            if set(row[side]) != set(names):\n                raise ValueError(\'Every declared landmark needs an explicit state on every frame\')\n            for name in names:\n                point(row[side][name], view)\n    phase_names = set()\n    for phase_doc in doc.get(\'phases\', []):\n        if phase_doc.get(\'name\') in phase_names:\n            raise ValueError(\'Duplicate phase name\')\n        phase_names.add(phase_doc.get(\'name\'))\n        text(phase_doc.get(\'name\'), \'phase name\')\n        bounds = phase_doc[\'window_frames\']\n        if len(bounds) != 2 or any(type(v) is not int for v in bounds) or not window[0] <= bounds[0] < bounds[1] <= window[1]:\n            raise ValueError(\'Phase bounds must be inside complete source window\')\n    events = doc.get(\'events\', [])\n    if not isinstance(events, list) or len(events) > 64:\n        raise ValueError(\'Invalid event list\')\n    seen = set()\n    pts_by_frame = {r[\'frame\']: r[\'source_pts\'] for r in rows}\n    for event in events:\n        name = text(event[\'name\'], \'event name\')\n        if name in seen:\n            raise ValueError(\'Duplicate event name\')\n        seen.add(name)\n        text(event[\'observation\'], \'visible event definition, not input-button inference\')\n        sf = event[\'source_frame_interval\']\n        ct = event.get(\'candidate_time_interval\')\n        if not isinstance(sf, list) or len(sf) != 2 or any(type(f) is not int or f not in pts_by_frame for f in sf) or sf[0] > sf[1]:\n            raise ValueError(\'Event source bracket must name observed window frames\')\n        if ct is not None and (len(ct) != 2 or rational(ct[0], \'event time\') > rational(ct[1], \'event time\')):\n            raise ValueError(\'Invalid candidate event bracket\')\n    return doc\n\ndef percentile(values, fraction):\n    values = sorted(values)\n    position = (len(values) - 1) * fraction\n    lo = math.floor(position); hi = math.ceil(position)\n    return values[lo] + (values[hi] - values[lo]) * (position - lo)\n\ndef stats(values):\n    if not values:\n        return None\n    return {\'rms\': math.sqrt(statistics.mean(v*v for v in values)), \'p95\': percentile(values, .95), \'max\': max(values), \'count\': len(values)}\n\ndef decision(low, high, threshold):\n    # Bounds touching a limit are inconclusive; these are annotation bounds, not CIs.\n    return \'within_threshold\' if high < threshold else \'outside_threshold\' if low > threshold else \'inconclusive\'\n\ndef interval_stats(errors, uncertainties, threshold):\n    lower = stats([max(0, e-u) for e, u in zip(errors, uncertainties)])\n    upper = stats([e+u for e, u in zip(errors, uncertainties)])\n    return {\'lower\': lower, \'upper\': upper, \'threshold\': threshold,\n            \'threshold_state\': decision(lower[\'rms\'], upper[\'rms\'], threshold) if lower and threshold is not None else \'unsupported\'}\n\ndef circular_delta(a, b):\n    return (a-b+180) % 360 - 180\n\ndef unwrapped(values):\n    result = []\n    for v in values:\n        result.append(v if not result else result[-1]+circular_delta(v, result[-1]))\n    return result\n\ndef compare(doc, supplied_artifacts=None, decoded_pts=None):\n    validate(doc)\n    artifacts = supplied_artifacts or {}\n    pts_witness = {\'verified_against_supplied_map\': False, \'source_video_bytes_verified\': False}\n    if decoded_pts is not None:\n        pts_bytes = decoded_pts if isinstance(decoded_pts, bytes) else json.dumps(decoded_pts,sort_keys=True,separators=(\',\',\':\')).encode()\n        pts_doc = strict_json(pts_bytes)\n        if \'time_base\' in pts_doc and rational(pts_doc[\'time_base\'],\'PTS map time_base\') != rational(doc[\'source\'][\'time_base\'],\'source time_base\'):\n            raise ValueError(\'Decoded PTS map time base differs from source declaration\')\n        if \'source_sha256\' in pts_doc and pts_doc[\'source_sha256\'] != doc[\'source\'][\'artifact\'][\'sha256\']:\n            raise ValueError(\'Decoded PTS map source identity differs\')\n        for row in doc[\'samples\']:\n            if row[\'frame\'] >= len(pts_doc[\'frames\']) or integer(pts_doc[\'frames\'][row[\'frame\']][\'pts\'],\'decoded map PTS\',-2**53) != row[\'source_pts\']:\n                raise ValueError(\'Source PTS differs from independently supplied decoded frame map\')\n        pts_witness = {\'verified_against_supplied_map\': True, \'source_video_bytes_verified\': False, \'map_sha256\': sha256(pts_bytes), \'decoded_frame_count\': len(pts_doc[\'frames\']), \'scope\':\'frame-index and integer-PTS matching only; pixel/annotation provenance not verified\', \'time_base_metadata_matched\': \'time_base\' in pts_doc, \'source_identity_metadata_matched\': \'source_sha256\' in pts_doc}\n    reg = doc[\'registration\']; ss = reg[\'source_scale\']; cs = reg[\'candidate_scale\']\n    L = doc[\'normalization\'][\'L_px\']; stride = doc[\'metrics_stride\']\n    rows = doc[\'samples\']; selected = rows[::stride]; names = [p[\'id\'] for p in doc[\'landmarks\']]\n    result = {\'schema\': \'rust-duty-weapon-diagnostics/v1\', \'status\': \'diagnostic\', \'action\': doc[\'action\'],\n        \'rubric_version\': doc[\'rubric_version\'], \'score\': None, \'artistic_approval\': False,\n        \'score_reason\': \'No percentage computed. Frozen event rubric and independent complete-clip review are separate requirements.\',\n        \'input_sha256\': sha256(json.dumps(doc, sort_keys=True, separators=(\',\', \':\'), allow_nan=False).encode()),\n        \'code_sha256\': sha256(Path(__file__).read_bytes()) if Path(__file__).is_file() else None,\n        \'registration\': reg, \'normalization\': doc[\'normalization\'],\n        \'landmark_definitions\': doc[\'landmarks\'], \'declared_annotation_review\': doc.get(\'annotation_review\'), \'source_window\': doc[\'source\'][\'window_frames\'],\n        \'metrics_stride\': stride, \'native_frame_count\': len(rows), \'metric_frame_count\': len(selected),\n        \'decoded_pts_witness\': pts_witness, \'provenance\': {}, \'landmarks\': {}, \'events\': [], \'hard_gate_findings\': [], \'unsupported\': [],\n        \'reviewer\': {\'status\': \'pending\', \'author\': doc[\'candidate\'][\'author\'], \'whole_clip_review\': None},\n        \'limits\': [\'2D screen projection does not establish 3D orientation, contact, anatomical or export/runtime correctness.\',\n                   \'Annotation uncertainty is a conservative bound, not a statistical confidence interval.\',\n                   \'No alignment is fitted, no points tracked automatically, no missing evidence awarded.\']}\n    for side in (\'source\', \'candidate\'):\n        art = doc[side][\'artifact\']; blob = artifacts.get(art[\'name\'])\n        if blob is not None and sha256(blob) != art[\'sha256\']:\n            raise ValueError(\'Supplied artifact hash mismatch: \' + art[\'name\'])\n        result[\'provenance\'][side] = {**art, \'bytes_verified\': blob is not None}\n    result[\'decoded_pts_witness\'][\'source_video_bytes_verified\'] = result[\'provenance\'][\'source\'][\'bytes_verified\']\n    pooled_raw = []; pooled_centered = []; pooled_uncertainty = []\n    baseline_row = next(r for r in rows if r[\'frame\'] == doc[\'baseline_frame\'])\n    worst = []\n    for name in names:\n        counts = {side: dict(Counter(r[side][name][\'state\'] for r in rows)) for side in (\'reference\', \'candidate\')}\n        missing = [r[\'frame\'] for r in rows if r[\'reference\'][name][\'state\'] == \'visible\' and r[\'candidate\'][name][\'state\'] != \'visible\']\n        if missing:\n            result[\'hard_gate_findings\'].append({\'kind\': \'candidate_not_observable_when_source_visible\', \'landmark\': name, \'frames\': missing})\n        usable = [r for r in selected if r[\'reference\'][name][\'state\'] == r[\'candidate\'][name][\'state\'] == \'visible\']\n        coverage = len(usable) / len(selected)\n        item = {\'native_state_counts\': counts, \'comparable_samples\': len(usable), \'scheduled_samples\': len(selected), \'coverage\': coverage,\n                \'native_coverage\': coverage_report(rows,name),\n                \'phase_coverage\': {p[\'name\']:coverage_report([r for r in rows if p[\'window_frames\'][0] <= r[\'frame\'] < p[\'window_frames\'][1]],name) for p in doc.get(\'phases\',[])},\n                \'raw\': None, \'mean_centered\': None, \'fixed_baseline_displacement\': None, \'amplitudes\': None}\n        if item[\'native_coverage\'][\'comparable_over_planned\'] < .8:\n            result[\'unsupported\'].append(name + \': native comparable coverage below 80%, inspect phase gaps\')\n        if coverage < .8:\n            result[\'unsupported\'].append(name + \': comparable sample coverage below 80%\')\n        if not usable:\n            result[\'landmarks\'][name] = item\n            continue\n        if len(usable) < 2:\n            errors = [math.dist([v*ss for v in r[\'reference\'][name][\'xy\']], [v*cs for v in r[\'candidate\'][name][\'xy\']]) for r in usable]\n            item[\'raw\'] = stats(errors)\n            result[\'unsupported\'].append(name + \': fewer than two comparable temporal samples\')\n            result[\'landmarks\'][name] = item\n            pooled_raw.extend(errors)\n            continue\n        a = [[v*ss for v in r[\'reference\'][name][\'xy\']] for r in usable]\n        b = [[v*cs for v in r[\'candidate\'][name][\'xy\']] for r in usable]\n        u = [r[\'reference\'][name][\'uncertainty_px\']*ss+r[\'candidate\'][name][\'uncertainty_px\']*cs for r in usable]\n        am = [statistics.mean(v[k] for v in a) for k in (0, 1)]\n        bm = [statistics.mean(v[k] for v in b) for k in (0, 1)]\n        raw = [math.dist(x,y) for x,y in zip(a,b)]\n        centered = [math.dist([x[k]-am[k] for k in (0,1)], [y[k]-bm[k] for k in (0,1)]) for x,y in zip(a,b)]\n        # Center uncertainty includes each point AND the uncertain constant mean.\n        centered_u = [v+statistics.mean(u) for v in u]\n        item.update(raw=stats(raw), mean_centered=stats(centered), constant_means={\'reference\': am, \'candidate\': bm},\n                    centered_uncertainty=interval_stats(centered, centered_u, .06*L if L is not None else None))\n        item[\'amplitudes\'] = {}\n        for k, axis in enumerate((\'x\',\'y\')):\n            ref = max(v[k] for v in a)-min(v[k] for v in a)\n            cand = max(v[k] for v in b)-min(v[k] for v in b)\n            err = abs(ref-cand)\n            bound = 2*(max(r[\'reference\'][name][\'uncertainty_px\']*ss for r in usable)+max(r[\'candidate\'][name][\'uncertainty_px\']*cs for r in usable))\n            threshold = max(.2*ref, .02*L) if L is not None else None\n            # Reference amplitude uncertainty also changes a relative tolerance.\n            ref_u = 2*max(r[\'reference\'][name][\'uncertainty_px\']*ss for r in usable)\n            low_tol = max(.2*max(0,ref-ref_u),.02*L) if L is not None else None\n            high_tol = max(.2*(ref+ref_u),.02*L) if L is not None else None\n            state = \'unsupported\' if L is None else \'within_threshold\' if err+bound < low_tol else \'outside_threshold\' if max(0,err-bound) > high_tol else \'inconclusive\'\n            item[\'amplitudes\'][axis] = {\'reference_px\': ref, \'candidate_px\': cand, \'error_px\': err,\n                 \'error_bound_px\': bound, \'threshold_px\': threshold, \'threshold_interval_px\': [low_tol,high_tol], \'threshold_state\': state}\n        ba = baseline_row[\'reference\'][name]; bb = baseline_row[\'candidate\'][name]\n        if ba[\'state\'] == bb[\'state\'] == \'visible\':\n            refbase = [v*ss for v in ba[\'xy\']]; candbase = [v*cs for v in bb[\'xy\']]\n            delta = [[(y[k]-candbase[k])-(x[k]-refbase[k]) for k in (0,1)] for x,y in zip(a,b)]\n            item[\'fixed_baseline_displacement\'] = {\'baseline_frame\': doc[\'baseline_frame\'], \'error\': stats([math.hypot(*v) for v in delta]),\n                                                 \'residual_xy_px\': delta, \'frames\': [r[\'frame\'] for r in usable],\n                                                 \'annotation_error_bound_px\': [v+ba[\'uncertainty_px\']*ss+bb[\'uncertainty_px\']*cs for v in u]}\n        else:\n            result[\'unsupported\'].append(name + \': pre-event baseline not observable\')\n        item[\'trace\'] = [{\'frame\': r[\'frame\'], \'source_pts\': r[\'source_pts\'], \'candidate_time\': r[\'candidate_time\'],\n                          \'raw_error_px\': e, \'centered_error_px\': ce, \'annotation_error_bound_px\': unc}\n                         for r,e,ce,unc in zip(usable, raw, centered, centered_u)]\n        worst.extend({\'frame\': r[\'frame\'], \'landmark\': name, \'raw_error_px\': e} for r,e in zip(usable,raw))\n        pooled_raw.extend(raw); pooled_centered.extend(centered); pooled_uncertainty.extend(centered_u)\n        result[\'landmarks\'][name] = item\n    result[\'aggregate\'] = {\'raw\': stats(pooled_raw), \'mean_centered\': stats(pooled_centered),\n        \'centered_uncertainty\': interval_stats(pooled_centered, pooled_uncertainty, .06*L if L is not None else None)}\n    if L is None:\n        result[\'unsupported\'].append(\'Reference normalization L unavailable; normalized spatial thresholds unsupported\')\n    result[\'worst_samples\'] = sorted(worst, key=lambda v:v[\'raw_error_px\'], reverse=True)[:24]\n    result[\'axis\'] = axis_report(doc, selected)\n    result[\'native_joins\'] = joins_report(doc, rows)\n    result[\'third_landmark_geometry\'] = third_landmark_report(doc, rows)\n    if result[\'axis\'].get(\'centered_status\',\'\').startswith(\'unsupported\'):\n        result[\'unsupported\'].append(result[\'axis\'][\'centered_status\'])\n    for side, values in result[\'third_landmark_geometry\'][\'sides\'].items():\n        if not values or all(v[\'noncollinearity\'] != \'supported\' for v in values):\n            result[\'unsupported\'].append(side + \': third weapon landmark has no observed noncollinear geometry\')\n    if result[\'axis\'][\'degenerate_frames\']:\n        result[\'unsupported\'].append(\'Degenerate projected weapon axis on listed frames\')\n    tb = rational(doc[\'source\'][\'time_base\'], \'time_base\')\n    by_frame = {r[\'frame\']: r for r in rows}\n    for event in doc.get(\'events\', []):\n        sr = [by_frame[f] for f in event[\'source_frame_interval\']]\n        expected = [rational(r[\'candidate_time\'], \'time\') for r in sr]\n        cand = event.get(\'candidate_time_interval\')\n        item = {\'name\': event[\'name\'], \'observation\': event[\'observation\'], \'source_frame_interval\': event[\'source_frame_interval\'],\n                \'mapped_source_time_interval\': [ratio(v) for v in expected], \'candidate_time_interval\': cand, \'phase_error_seconds_interval\': None}\n        if cand is not None:\n            c = [rational(v, \'candidate event time\') for v in cand]\n            item[\'phase_error_seconds_interval\'] = [float(c[0]-expected[1]),float(c[1]-expected[0])]\n        else:\n            result[\'unsupported\'].append(\'Candidate event missing: \' + event[\'name\'])\n        result[\'events\'].append(item)\n    result[\'native_source_duration_between_samples_seconds\'] = ratio((rows[-1][\'source_pts\']-rows[0][\'source_pts\'])*tb)\n    return result\n\ndef coverage_report(rows,name):\n    visible=[r for r in rows if r[\'reference\'][name][\'state\']==\'visible\']\n    valid=[r for r in visible if r[\'candidate\'][name][\'state\']==\'visible\']\n    gaps=[];start=None;last=None\n    for r in rows:\n        bad=r[\'reference\'][name][\'state\']!=\'visible\' or r[\'candidate\'][name][\'state\']!=\'visible\'\n        if bad:\n            if start is None:start=r[\'frame\']\n            last=r[\'frame\']\n        elif start is not None:\n            gaps.append([start,last+1]);start=None\n    if start is not None:gaps.append([start,last+1])\n    return {\'planned\':len(rows),\'source_visible\':len(visible),\'candidate_valid_when_source_visible\':len(valid),\n            \'source_visible_over_planned\':len(visible)/len(rows),\n            \'candidate_valid_over_source_visible\':len(valid)/len(visible) if visible else None,\n            \'comparable_over_planned\':len(valid)/len(rows),\'noncomparable_intervals_half_open\':gaps}\n\ndef axis_report(doc, rows):\n    ids = doc[\'axis_landmarks\']; reg = doc[\'registration\']; angles = [[],[]]; side_bounds = [[],[]]; bounds = []; frames = []; degenerate = []\n    for r in rows:\n        if not all(r[side][n][\'state\'] == \'visible\' for side in (\'reference\',\'candidate\') for n in ids):\n            continue\n        local = []; unc = []\n        for side in (\'reference\',\'candidate\'):\n            a,b = [r[side][n] for n in ids]\n            d = [b[\'xy\'][k]-a[\'xy\'][k] for k in (0,1)]; length = math.hypot(*d)\n            u = a[\'uncertainty_px\']+b[\'uncertainty_px\']\n            if length <= max(u, 1e-8):\n                degenerate.append(r[\'frame\']); break\n            local.append(math.degrees(math.atan2(d[1],d[0])))\n            unc.append(math.degrees(math.asin(min(1,u/length))))\n        if len(local) == 2:\n            for i in (0,1):\n                angles[i].append(local[i]); side_bounds[i].append(unc[i])\n            frames.append(r[\'frame\']); bounds.append(sum(unc))\n    report = {\'frames\':frames,\'degenerate_frames\':sorted(set(degenerate)), \'raw_error_degrees\':None,\'centered_error_degrees\':None,\'coverage\':len(frames)/len(rows)}\n    if not frames: return report\n    if len(frames) < 2:\n        report[\'raw_error_degrees\'] = stats([abs(circular_delta(x,y)) for x,y in zip(*angles)])\n        report[\'centered_status\']=\'unsupported: fewer than two comparable temporal samples\'\n        return report\n    raw = [abs(circular_delta(x,y)) for x,y in zip(*angles)]\n    report[\'raw_error_degrees\'] = stats(raw)\n    branch_risks=[]\n    native_by_frame = {r[\'frame\']:r for r in doc[\'samples\']}\n    for i in range(1,len(frames)):\n        hidden_between = any(any(native_by_frame[f][side][n][\'state\']!=\'visible\' for side in (\'reference\',\'candidate\') for n in ids) for f in range(frames[i-1]+1,frames[i]))\n        if hidden_between or frames[i]-frames[i-1] != doc[\'metrics_stride\']:\n            branch_risks.append({\'from_frame\':frames[i-1],\'to_frame\':frames[i],\'reason\':\'visibility gap\'})\n        for side in (0,1):\n            if abs(circular_delta(angles[side][i],angles[side][i-1]))+side_bounds[side][i]+side_bounds[side][i-1] >= 180:\n                branch_risks.append({\'from_frame\':frames[i-1],\'to_frame\':frames[i],\'reason\':\'uncertainty crosses unwrap branch\',\'side\':(\'reference\',\'candidate\')[side]})\n    report[\'unwrap_unsupported_intervals\']=branch_risks\n    if branch_risks:\n        report[\'centered_status\']=\'unsupported: ambiguous unwrap branch or visibility gap\'\n        return report\n    a,b = map(unwrapped, angles)\n    am=statistics.mean(a); bm=statistics.mean(b)\n    centered=[abs((x-am)-(y-bm)) for x,y in zip(a,b)]\n    amp_a=max(a)-min(a); amp_b=max(b)-min(b); amp_err=abs(amp_a-amp_b)\n    report.update(raw_error_degrees=stats(raw),centered_error_degrees=stats(centered),\n                  constant_means_degrees=[am,bm],reference_trace_degrees=a,candidate_trace_degrees=b,\n                  centered_uncertainty=interval_stats(centered,[u+statistics.mean(bounds) for u in bounds],5),\n                  amplitude={\'reference_degrees\':amp_a,\'candidate_degrees\':amp_b,\'error_degrees\':amp_err,\n                             \'annotation_error_bound_degrees\':2*(max(side_bounds[0])+max(side_bounds[1])),\'threshold_degrees\':max(.2*amp_a,2)},\n                  direction_agreement_diagnostic=sum((x-am)*(y-bm) for x,y in zip(a,b)))\n    return report\n\ndef joins_report(doc, rows):\n    # Native adjacent displacement/speed; no invented continuity points or loop wrap.\n    output={}; tb=rational(doc[\'source\'][\'time_base\'],\'time_base\')\n    for name in (p[\'id\'] for p in doc[\'landmarks\']):\n        sides={}\n        for side,scale in ((\'reference\',doc[\'registration\'][\'source_scale\']),(\'candidate\',doc[\'registration\'][\'candidate_scale\'])):\n            values=[]\n            for a,b in zip(rows,rows[1:]):\n                if a[side][name][\'state\']==b[side][name][\'state\']==\'visible\':\n                    d=math.dist(a[side][name][\'xy\'],b[side][name][\'xy\'])*scale\n                    dt=float((b[\'source_pts\']-a[\'source_pts\'])*tb)\n                    bound=(a[side][name][\'uncertainty_px\']+b[side][name][\'uncertainty_px\'])*scale\n                    values.append({\'from_frame\':a[\'frame\'],\'to_frame\':b[\'frame\'],\'displacement_px\':d,\'speed_px_s\':d/dt,\'annotation_displacement_bound_px\':bound,\'annotation_speed_bound_px_s\':bound/dt})\n            sides[side]={\'largest_steps\':sorted(values,key=lambda r:r[\'displacement_px\'],reverse=True)[:12],\n                         \'largest_speeds\':sorted(values,key=lambda r:r[\'speed_px_s\'],reverse=True)[:12],\n                         \'observed_adjacent_pairs\':len(values),\'planned_adjacent_pairs\':len(rows)-1,\n                         \'step_stats\':stats([v[\'displacement_px\'] for v in values]), \'speed_stats\':stats([v[\'speed_px_s\'] for v in values])}\n        output[name]=sides\n    return output\n\ndef third_landmark_report(doc, rows):\n    names=[p[\'id\'] for p in doc[\'landmarks\']]; a,b=doc[\'axis_landmarks\']; third=next(n for n in names if n not in (a,b))\n    result={\'third_landmark\':third,\'scope\':\'Signed screen triangle area is a direction/degeneracy witness, not 3D orientation proof.\',\'sides\':{}}\n    for side in (\'reference\',\'candidate\'):\n        areas=[]\n        for r in rows:\n            if all(r[side][n][\'state\']==\'visible\' for n in (a,b,third)):\n                p,q,t=[r[side][n][\'xy\'] for n in (a,b,third)]\n                area=(q[0]-p[0])*(t[1]-p[1])-(q[1]-p[1])*(t[0]-p[0])\n                up,uq,ut=[r[side][n][\'uncertainty_px\'] for n in (a,b,third)]\n                du=up+uq;dv=up+ut\n                area_bound=du*math.dist(t,p)+dv*math.dist(q,p)+du*dv\n                areas.append({\'frame\':r[\'frame\'],\'signed_double_area_px2\':area,\'annotation_area_bound_px2\':area_bound,\'noncollinearity\':\'supported\' if abs(area)>area_bound else \'inconclusive\'})\n        result[\'sides\'][side]=areas\n    return result\n\ndef plot_report(report, output):\n    import matplotlib.pyplot as plt\n    fig,axes=plt.subplots(2,1,figsize=(11,7),sharex=True)\n    for name,item in report[\'landmarks\'].items():\n        trace=item.get(\'trace\',[])\n        if not trace: continue\n        axes[0].plot([r[\'frame\'] for r in trace],[r[\'raw_error_px\'] for r in trace],label=name)\n        axes[1].plot([r[\'frame\'] for r in trace],[r[\'centered_error_px\'] for r in trace],label=name)\n    axes[0].set_ylabel(\'Raw error, pixels\'); axes[1].set_ylabel(\'Mean-centered error, pixels\'); axes[1].set_xlabel(\'Exact source frame\')\n    if report[\'normalization\'][\'L_px\'] is not None:\n        axes[1].axhline(.06*report[\'normalization\'][\'L_px\'],color=\'black\',linestyle=\'--\',label=\'0.06 L diagnostic limit\')\n    for ax in axes: ax.legend()\n    fig.suptitle(report[\'action\']+\' / diagnostic only / no score\'); fig.tight_layout(); fig.savefig(output,dpi=130); plt.close(fig)\n\ndef main():\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'input\',type=Path); parser.add_argument(\'--output\',type=Path,required=True); parser.add_argument(\'--plot\',action=\'store_true\'); parser.add_argument(\'--pts-map\',type=Path)\n    args=parser.parse_args(); blob=args.input.read_bytes(); doc=strict_json(blob); result=compare(doc,decoded_pts=args.pts_map.read_bytes() if args.pts_map else None)\n    result[\'input_file_sha256\']=sha256(blob)\n    args.output.mkdir(parents=True,exist_ok=True)\n    (args.output/\'report.json\').write_text(json.dumps(result,indent=2,allow_nan=False)+\'\\n\')\n    if args.plot: plot_report(result,args.output/\'residuals.png\')\n    print(json.dumps({\'status\':result[\'status\'],\'action\':result[\'action\'],\'score\':result[\'score\'],\'hard_gate_findings\':len(result[\'hard_gate_findings\']),\'output\':str(args.output)}))\n\nif __name__==\'__main__\': main()\n'
module=types.ModuleType('weapon_compare'); module.__file__='<embedded:weapon_compare>'
sys.modules['weapon_compare']=module
exec(compile(MODULE_SOURCE,module.__file__,'exec'),module.__dict__)
CODE_SHA256=hashlib.sha256(MODULE_SOURCE.encode()).hexdigest()
print('Module SHA256:',CODE_SHA256)


In [ ]:
DEMO=True # False: upload one bounded comparison JSON; do not upload private assets.
DECODED_PTS_JSON='' # Optional exact filename of independently decoded ffprobe frame map.
if DEMO:
    INPUTS={'example.synthetic.json':b'{\n  "schema": "rust-duty-weapon-comparison/v1",\n  "action": "synthetic-event",\n  "kind": "event",\n  "rubric_version": "event-amendment-pending",\n  "source": {\n    "artifact": {\n      "name": "source.mp4",\n      "sha256": "0000000000000000000000000000000000000000000000000000000000000000"\n    },\n    "viewport": [\n      1280,\n      720\n    ],\n    "time_base": [\n      1,\n      15360\n    ],\n    "window_frames": [\n      100,\n      112\n    ]\n  },\n  "candidate": {\n    "artifact": {\n      "name": "candidate.blend",\n      "sha256": "1111111111111111111111111111111111111111111111111111111111111111"\n    },\n    "viewport": [\n      1280,\n      720\n    ],\n    "space": "evaluated-camera-pixels",\n    "author": "synthetic author"\n  },\n  "registration": {\n    "mode": "frozen-unwarped",\n    "source_scale": 1,\n    "candidate_scale": 1,\n    "source_anchor_pts": 25600,\n    "candidate_anchor_time": [\n      0,\n      1\n    ],\n    "phase_offset": [\n      0,\n      1\n    ],\n    "frozen_record": "synthetic fixture only"\n  },\n  "normalization": {\n    "L_px": 250,\n    "definition": "synthetic fixed axis length"\n  },\n  "metrics_stride": 3,\n  "baseline_frame": 100,\n  "landmarks": [\n    {\n      "id": "muzzle",\n      "physical_point": "center of muzzle opening"\n    },\n    {\n      "id": "receiver",\n      "physical_point": "top rear receiver corner"\n    },\n    {\n      "id": "sight",\n      "physical_point": "top sight center"\n    }\n  ],\n  "axis_landmarks": [\n    "muzzle",\n    "receiver"\n  ],\n  "samples": [\n    {\n      "frame": 100,\n      "source_pts": 25600,\n      "candidate_time": [\n        0,\n        60\n      ],\n      "reference": {\n        "muzzle": {\n          "xy": [\n            100,\n            100\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "receiver": {\n          "xy": [\n            350,\n            100\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "sight": {\n          "xy": [\n            325,\n            80\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        }\n      },\n      "candidate": {\n        "muzzle": {\n          "xy": [\n            100,\n            100\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "receiver": {\n          "xy": [\n            350,\n            100\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "sight": {\n          "xy": [\n            325,\n            80\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        }\n      }\n    },\n    {\n      "frame": 101,\n      "source_pts": 25856,\n      "candidate_time": [\n        1,\n        60\n      ],\n      "reference": {\n        "muzzle": {\n          "xy": [\n            102,\n            101\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "receiver": {\n          "xy": [\n            352,\n            101\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "sight": {\n          "xy": [\n            327,\n            81\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        }\n      },\n      "candidate": {\n        "muzzle": {\n          "xy": [\n            102,\n            101\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "receiver": {\n          "xy": [\n            352,\n            101\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "sight": {\n          "xy": [\n            327,\n            81\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        }\n      }\n    },\n    {\n      "frame": 102,\n      "source_pts": 26112,\n      "candidate_time": [\n        2,\n        60\n      ],\n      "reference": {\n        "muzzle": {\n          "xy": [\n            104,\n            102\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "receiver": {\n          "xy": [\n            354,\n            102\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "sight": {\n          "xy": [\n            329,\n            82\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        }\n      },\n      "candidate": {\n        "muzzle": {\n          "xy": [\n            104,\n            102\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "receiver": {\n          "xy": [\n            354,\n            102\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "sight": {\n          "xy": [\n            329,\n            82\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        }\n      }\n    },\n    {\n      "frame": 103,\n      "source_pts": 26368,\n      "candidate_time": [\n        3,\n        60\n      ],\n      "reference": {\n        "muzzle": {\n          "xy": [\n            106,\n            103\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "receiver": {\n          "xy": [\n            356,\n            103\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "sight": {\n          "xy": [\n            331,\n            83\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        }\n      },\n      "candidate": {\n        "muzzle": {\n          "xy": [\n            106,\n            103\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "receiver": {\n          "xy": [\n            356,\n            103\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "sight": {\n          "xy": [\n            331,\n            83\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        }\n      }\n    },\n    {\n      "frame": 104,\n      "source_pts": 26624,\n      "candidate_time": [\n        4,\n        60\n      ],\n      "reference": {\n        "muzzle": {\n          "xy": [\n            108,\n            104\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "receiver": {\n          "xy": [\n            358,\n            104\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "sight": {\n          "xy": [\n            333,\n            84\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        }\n      },\n      "candidate": {\n        "muzzle": {\n          "xy": [\n            108,\n            104\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "receiver": {\n          "xy": [\n            358,\n            104\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "sight": {\n          "xy": [\n            333,\n            84\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        }\n      }\n    },\n    {\n      "frame": 105,\n      "source_pts": 26880,\n      "candidate_time": [\n        5,\n        60\n      ],\n      "reference": {\n        "muzzle": {\n          "xy": [\n            110,\n            105\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "receiver": {\n          "xy": [\n            360,\n            105\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "sight": {\n          "xy": [\n            335,\n            85\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        }\n      },\n      "candidate": {\n        "muzzle": {\n          "xy": [\n            110,\n            105\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "receiver": {\n          "xy": [\n            360,\n            105\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "sight": {\n          "xy": [\n            335,\n            85\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        }\n      }\n    },\n    {\n      "frame": 106,\n      "source_pts": 27136,\n      "candidate_time": [\n        6,\n        60\n      ],\n      "reference": {\n        "muzzle": {\n          "xy": [\n            112,\n            106\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "receiver": {\n          "xy": [\n            362,\n            106\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "sight": {\n          "xy": [\n            337,\n            86\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        }\n      },\n      "candidate": {\n        "muzzle": {\n          "xy": [\n            112,\n            106\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "receiver": {\n          "xy": [\n            362,\n            106\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "sight": {\n          "xy": [\n            337,\n            86\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        }\n      }\n    },\n    {\n      "frame": 107,\n      "source_pts": 27392,\n      "candidate_time": [\n        7,\n        60\n      ],\n      "reference": {\n        "muzzle": {\n          "xy": [\n            114,\n            107\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "receiver": {\n          "xy": [\n            364,\n            107\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "sight": {\n          "xy": [\n            339,\n            87\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        }\n      },\n      "candidate": {\n        "muzzle": {\n          "xy": [\n            114,\n            107\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "receiver": {\n          "xy": [\n            364,\n            107\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "sight": {\n          "xy": [\n            339,\n            87\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        }\n      }\n    },\n    {\n      "frame": 108,\n      "source_pts": 27648,\n      "candidate_time": [\n        8,\n        60\n      ],\n      "reference": {\n        "muzzle": {\n          "xy": [\n            116,\n            108\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "receiver": {\n          "xy": [\n            366,\n            108\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "sight": {\n          "xy": [\n            341,\n            88\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        }\n      },\n      "candidate": {\n        "muzzle": {\n          "xy": [\n            116,\n            108\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "receiver": {\n          "xy": [\n            366,\n            108\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "sight": {\n          "xy": [\n            341,\n            88\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        }\n      }\n    },\n    {\n      "frame": 109,\n      "source_pts": 27904,\n      "candidate_time": [\n        9,\n        60\n      ],\n      "reference": {\n        "muzzle": {\n          "xy": [\n            118,\n            109\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "receiver": {\n          "xy": [\n            368,\n            109\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "sight": {\n          "xy": [\n            343,\n            89\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        }\n      },\n      "candidate": {\n        "muzzle": {\n          "xy": [\n            118,\n            109\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "receiver": {\n          "xy": [\n            368,\n            109\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "sight": {\n          "xy": [\n            343,\n            89\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        }\n      }\n    },\n    {\n      "frame": 110,\n      "source_pts": 28160,\n      "candidate_time": [\n        10,\n        60\n      ],\n      "reference": {\n        "muzzle": {\n          "xy": [\n            120,\n            110\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "receiver": {\n          "xy": [\n            370,\n            110\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "sight": {\n          "xy": [\n            345,\n            90\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        }\n      },\n      "candidate": {\n        "muzzle": {\n          "xy": [\n            120,\n            110\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "receiver": {\n          "xy": [\n            370,\n            110\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "sight": {\n          "xy": [\n            345,\n            90\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        }\n      }\n    },\n    {\n      "frame": 111,\n      "source_pts": 28416,\n      "candidate_time": [\n        11,\n        60\n      ],\n      "reference": {\n        "muzzle": {\n          "xy": [\n            122,\n            111\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "receiver": {\n          "xy": [\n            372,\n            111\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "sight": {\n          "xy": [\n            347,\n            91\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        }\n      },\n      "candidate": {\n        "muzzle": {\n          "xy": [\n            122,\n            111\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "receiver": {\n          "xy": [\n            372,\n            111\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        },\n        "sight": {\n          "xy": [\n            347,\n            91\n          ],\n          "state": "visible",\n          "uncertainty_px": 0.1,\n          "method": "synthetic known coordinates"\n        }\n      }\n    }\n  ],\n  "events": [\n    {\n      "name": "dip",\n      "observation": "visible minimum, synthetic only",\n      "source_frame_interval": [\n        105,\n        106\n      ],\n      "candidate_time_interval": [\n        [\n          5,\n          60\n        ],\n        [\n          6,\n          60\n        ]\n      ]\n    }\n  ]\n}\n'}
    INPUT_JSON='example.synthetic.json'
else:
    from google.colab import files
    INPUTS=files.upload()
    if not 1 <= len(INPUTS) <= 2 or any(not n.endswith('.json') or len(b)>module.MAX_BYTES for n,b in INPUTS.items()):
        raise ValueError('Select comparison JSON and optional decoded PTS JSON only, max32MiB each')
    INPUT_JSON=next(n for n in INPUTS if n!=DECODED_PTS_JSON)
    if len(INPUTS)==2 and not DECODED_PTS_JSON:
        raise ValueError('Set DECODED_PTS_JSON to the exact second file name before running')
    print('Selected comparison:',INPUT_JSON)


In [ ]:
doc=module.strict_json(INPUTS[INPUT_JSON])
pts=INPUTS[DECODED_PTS_JSON] if DECODED_PTS_JSON else None
result=module.compare(doc,decoded_pts=pts)
result['code_sha256']=CODE_SHA256
result['input_file_sha256']=module.sha256(INPUTS[INPUT_JSON])
result['execution']={'backend':'CPU','synthetic':DEMO,'scope':'weapon measurements only'}
OUTPUT=Path(tempfile.mkdtemp(prefix='weapon-motion-'))
(OUTPUT/'report.json').write_text(json.dumps(result,indent=2,allow_nan=False)+'\n')
module.plot_report(result,OUTPUT/'residuals.png')
from IPython.display import display,Image
display(Image(filename=str(OUTPUT/'residuals.png')))
print(json.dumps({'action':result['action'],'score':result['score'],'unsupported':result['unsupported'],'hard_gate_findings':result['hard_gate_findings']},indent=2))


In [ ]:
archive=OUTPUT.parent/(OUTPUT.name+'.zip')
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUTPUT.iterdir()):z.write(p,p.name)
ARCHIVE_SHA256=module.sha256(archive.read_bytes())
print('Result ZIP SHA256:',ARCHIVE_SHA256)
from google.colab import files
files.download(str(archive))


Save and independently hash-check the downloaded ZIP before releasing a runtime.
A download request does not prove durable preservation. After saving, use
Runtime → Disconnect and delete runtime; verify no active entry remains in
Manage sessions. No automatic GPU allocation or automatic teardown is performed.
All source/candidate artifact hashes remain declared unless their bytes were
verified separately. This notebook verifies an optional PTS map, not video pixels.
